## **LangChain** - _Text Splitters_

Los **Text Splitters** son herramientas diseñadas para dividir documentos largos en fragmentos más pequeños y manejables, conocidos como _chunks_.

Es un paso fundamental cuando trabajas con **LLMs** por dos razones principales:
- **Límites de la Ventana de Contexto**: Los LLMs tienen un límite en la cantidad de texto que pueden procesar a la vez (medido en tokens). No puedes enviarle un libro entero de 500 páginas en un solo prompt.

- **Sistemas RAG**: Si quieres que una IA responda preguntas basándose en tus propios documentos, primero debes guardar esos documentos en una base de datos vectorial. Guardar fragmentos pequeños y precisos permite que el sistema encuentre exactamente el párrafo que contiene la respuesta, en lugar de intentar procesar todo el documento original de golpe.

```bash
pip install langchain_text_splitters tiktoken
```

### 01. Splitting

Es el proceso mediante el cual **LangChain** toma un documento largo y lo divide en fragmentos más pequeños, conocidos como _chunks_.


Cuando configuras un **Text Splitter** en **LangChain**, prácticamente todo se controla a través de dos parámetros fundamentales:

- `chunk_size` (_Tamaño del Fragmento_):
    - Define el límite máximo de tamaño que tendrá cada pedazo de texto. Dependiendo del divisor que elijas, este tamaño se puede medir en número de caracteres o en número de tokens.
    - Si el tamaño es muy grande, podrías saturar al **LLM**; si es muy pequeño, el fragmento podría perder su significado.

- `chunk_overlap` (_Superposición_):
    - Determina la cantidad de texto (caracteres o tokens) que se repetirá entre el final de un fragmento y el principio del siguiente.
    - Este parámetro es vital para no perder el contexto. Actúa como un puente lógico para evitar que una oración importante o un concepto clave quede cortado por la mitad y pierda todo el sentido.

### 02. **RecursiveCharacterTextSplitter**

Este es el divisor recomendado para casi cualquier texto general.

Es "recursivo" porque intenta dividir el texto usando una lista de separadores en orden jerárquico. Si un párrafo es demasiado grande, intentará dividirlo por oraciones (saltos de línea), luego por palabras (espacios), y finalmente por letras, intentando siempre mantener juntas las ideas.

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Texto de prueba
texto_largo = """El procesamiento del lenguaje natural es un campo de la IA.

LangChain facilita la conexión de modelos de lenguaje con otras fuentes de datos. Esto es crucial para sistemas RAG.

Dividir el texto correctamente ayuda a mantener el contexto semántico."""

# Splitter
divisor_recursivo = RecursiveCharacterTextSplitter(chunk_size = 60,       # Tamaño máximo de caracteres por fragmento
                                                   chunk_overlap = 10,    # Cuántos caracteres se superponen
                                                   separators = ["\n\n", "\n", " ", ""] # Orden de separadores
                                                   )

# Ejecutar la división
fragmentos = divisor_recursivo.split_text(text = texto_largo)

for i, fragmento in enumerate(fragmentos):
    print(f"--- Chunk {i+1} ---")
    print(fragmento)

--- Chunk 1 ---
El procesamiento del lenguaje natural es un campo de la IA.
--- Chunk 2 ---
LangChain facilita la conexión de modelos de lenguaje con
--- Chunk 3 ---
con otras fuentes de datos. Esto es crucial para sistemas
--- Chunk 4 ---
sistemas RAG.
--- Chunk 5 ---
Dividir el texto correctamente ayuda a mantener el contexto
--- Chunk 6 ---
contexto semántico.


### 03. **CharacterTextSplitter**

Este divisor es mucho más simple y menos flexible.

Divide el texto basándose en un único carácter que le especifiques (por defecto, un doble salto de línea `\n\n`).

Es útil si tienes un documento muy estructurado donde sabes exactamente qué carácter separa las secciones.

In [6]:
from langchain_text_splitters import CharacterTextSplitter

texto_estructurado = "Sección 1: Inicio del documento.\n\nSección 2: Desarrollo del tema.\n\nSección 3: Conclusión final."

# Splitter
divisor_caracter = CharacterTextSplitter(separator = "\n\n",    # Separador estricto
                                         chunk_size = 35,
                                         chunk_overlap = 0)

# Ejecutar la división
fragmentos = divisor_caracter.split_text(text = texto_estructurado)

for i, fragmento in enumerate(fragmentos):
    print(f"--- Chunk {i+1} ---")
    print(fragmento)

--- Chunk 1 ---
Sección 1: Inicio del documento.
--- Chunk 2 ---
Sección 2: Desarrollo del tema.
--- Chunk 3 ---
Sección 3: Conclusión final.


### 04. **TokenTextSplitter**

Los Modelos de Lenguaje no leen "caracteres", leen "tokens" (que suelen ser fragmentos de palabras).

Si necesitas estar seguro de que un fragmento va a caber en la ventana de contexto de un **LLM**, debes dividir el texto por tokens.

Este divisor utiliza la librería `tiktoken` (creada por **OpenAI**) para hacer el cálculo exacto.

In [8]:
from langchain_text_splitters import TokenTextSplitter

texto_tecnico = "En este ejemplo vamos a dividir el texto usando la medida exacta que entienden los modelos como GPT-4: los tokens."

# Splitter
divisor_tokens = TokenTextSplitter(chunk_size = 10,       # Tamaño máximo en TOKENS, no en caracteres
                                   chunk_overlap = 2      # Superposición en TOKENS
                                   )

# Ejecutar la división
fragmentos = divisor_tokens.split_text(text = texto_tecnico)

for i, fragmento in enumerate(fragmentos):
    print(f"--- Chunk {i+1} ---")
    print(fragmento)

--- Chunk 1 ---
En este ejemplo vamos a
--- Chunk 2 ---
os a dividir el texto usando la
--- Chunk 3 ---
ando la medida exacta que entiend
--- Chunk 4 ---
ienden los modelos como GPT
--- Chunk 5 ---
 GPT-4: los tokens.
